In [ ]:
import copy
import csv
import glob
import json
import librosa
import librosa.display
import random
import os
import sys
import torch
import ast  # for parsing stringified tuples in CSV


import IPython.display as ipd
from IPython.display import clear_output
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from collections import defaultdict
from scipy.io import wavfile
from scipy.spatial.distance import euclidean
from scipy import stats

sys.path.append("/om2/user/gelbanna/repos/projects/continuous-speech-recognition")

from front_end.cochlear_model import CochlearModel
from utils import load_yaml_config

#testing out cochlear model

In [ ]:
# Example instantiation
config = load_yaml_config("/om2/user/bjmedina/memory/utils/cochleagram_params.yaml")
config['frontend']['cochlear']['filter_params']['sr'] = 44100
model = CochlearModel(config.frontend.cochlear, device="cuda")


# Example forward pass
# x = [torch.randn(1, 1, 16000), torch.randn(1, 1, 8000), torch.randn(1, 1, 32000)]  # Example input tensor
# output = model(x[0].to("cuda"))
# print(output.shape)  # Example output shape 

In [ ]:
config

In [ ]:
def plot_torch_cochleagram(output):
    cochleagram_dB = output.squeeze(0).squeeze(0).detach().cpu().numpy()
    
    plt.figure(figsize=(10, 6))
    plt.imshow(
        cochleagram_dB,
        origin='lower',
        aspect='auto',
        # extent=[time_bins[0].item(), time_bins[-1].item(),
        #         freq_bins[0].item(), freq_bins[-1].item()],
        cmap='viridis'
    )
    plt.colorbar(label='Amplitude [dB]')
    plt.title("Cochleagram")
    plt.xlabel("Time [s]")
    plt.ylabel("Frequency [Hz]")
    plt.tight_layout()
    plt.show()
    
def plot_torch_corrs(output):
    corrs = output.squeeze(0).squeeze(0).detach().cpu().numpy()
    
    plt.figure(figsize=(10, 6))
    plt.matshow(
        corrs, vmin=0, vmax=1
    )
    plt.colorbar(label='Pearson Correlation')
    plt.title("Segments")
    plt.xlabel("Segments")
    plt.ylabel("Segments")
    plt.tight_layout()
    plt.show()
# plot_torch_cochleagram(output)

In [ ]:
def rms_normalize(signal, target_rms=0.1):
    """
    Normalize a signal to a target RMS level.
    
    Parameters:
        signal (numpy array): The input signal.
        target_rms (float): The desired RMS level (default is 0.1).
    
    Returns:
        numpy array: The RMS-normalized signal.
    """
    rms = np.sqrt(np.mean(signal**2))
    if rms == 0:
        return signal  # Avoid division by zero
    return signal * (target_rms / rms)

In [ ]:
# grabbing stationarity scores for sounds that are in the in the eval set of AudioSet (which is what we typically use for memory experiment)
stationarity_scores_path = "/om2/user/bjmedina/chexture_choolbox/assets/OVERLAPPED_0.5s_eval_sound_list_with_stationarity_score_no_speech_no_music_audioset_matlab_coch_rms0p02.csv"
stationarity_scores_ = pd.read_csv(stationarity_scores_path)

In [ ]:
len(set(stationarity_scores_.filepath.tolist()))

In [ ]:
# grabbing the textures that were selected to be in the memory 
texture_pairs_paths = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exemplar_0.5s_adjacent/texture_filenames.json"

with open(texture_pairs_paths) as f:
    texture_pairs = json.load(f)

In [ ]:
len(texture_pairs)

In [ ]:
ss_scores_to_texture = defaultdict(list)
times_to_texture     = defaultdict(list)

for texture_pair in texture_pairs:
    texture_id = "/".join(texture_pair['full_path'].split("/")[-2:])

    avg_ss_scores = stationarity_scores_[stationarity_scores_.filepath==texture_id].stationarity_score.tolist()
    times = stationarity_scores_[stationarity_scores_.filepath==texture_id].onset_time.tolist()
    
    ss_scores_to_texture[texture_id] = avg_ss_scores
    times_to_texture[texture_id]     = times

In [ ]:
# plot distribution of stationarity scores
avg_scores = []
for id in ss_scores_to_texture:
    avg_scores.extend(ss_scores_to_texture[id])

In [ ]:
plt.title("Stationarity Scores")
plt.axvline(x=np.mean(avg_scores), label=f"$\mu={np.mean(avg_scores)}$\n$\sigma={np.std(avg_scores)}$")
plt.hist(avg_scores, bins=100, alpha=0.5)
plt.legend()

avg_ss_score = np.mean(avg_scores)
std_ss_score = np.std(avg_scores)

In [ ]:
avg_ss_score

In [ ]:
# consider how you screen for different sound
fps = stationarity_scores_[stationarity_scores_.stationarity_score < avg_ss_score].filepath.tolist(); fps = list(set(fps))

In [ ]:
len(fps)

In [ ]:
base_dir = "/om2/data/public/audioset/wavs/eval_segments_downloads/"
offset = 0.1

save_dir = f"/om2/user/bjmedina/data/texture_pairs/cochleagram_norms_ratio-max_stationarity-lessthan{avg_ss_score:.2f}-offset{offset}/"


In [ ]:
import itertools

def abs_then_normalize(arr):
    """Min-max normalize an array to range [0,1]."""
    arr = np.abs(arr)  # Take absolute values first
    min_val, max_val = np.min(arr), np.max(arr)
    return (arr - min_val) / (max_val - min_val) if max_val > min_val else arr


def find_best_stationarity_match(times, stationarity_scores, target_stationarity):
    """
    Finds the time index with the closest stationarity score to the target.

    Args:
        times (list or array): List of time values.
        stationarity_scores (list or array): List of stationarity scores corresponding to times.
        target_stationarity (float): The desired stationarity score to match.

    Returns:
        tuple: (best_x, best_time, best_ss) where
            - best_x is the index of the closest stationarity score,
            - best_time is the corresponding time,
            - best_ss is the closest stationarity score.
    """
    min_mse = float('inf')  # Initialize with a very large number
    best_x = -1
    best_time = 0
    best_ss = 0

    for x, time in enumerate(times): 
        mse = (target_stationarity - stationarity_scores[x]) ** 2

        if mse < min_mse:
            best_x = x
            best_time = time
            min_mse = mse
            best_ss = stationarity_scores[x]

    return best_x, best_time, best_ss

def find_max_stationarity(times, stationarity_scores, target_stationarity):
    """
    Finds the time index with the closest stationarity score to the target.

    Args:
        times (list or array): List of time values.
        stationarity_scores (list or array): List of stationarity scores corresponding to times.
        target_stationarity (float): The desired stationarity score to match.

    Returns:
        tuple: (best_x, best_time, best_ss) where
            - best_x is the index of the closest stationarity score,
            - best_time is the corresponding time,
            - best_ss is the max stationarity score (least stationary sound).
    """
    max_mse = -float('inf')  # Initialize with a very large number
    best_x = -1
    best_time = 0
    best_ss = 0

    for x, time in enumerate(times): 
        mse = stationarity_scores[x]

        if mse > max_mse:
            best_x = x
            best_time = time
            min_mse = mse
            best_ss = stationarity_scores[x]

    return best_x, best_time, best_ss

def apply_linear_ramp(signal, sample_rate, ramp_duration_ms=5):
    """
    Applies a linear ramp (fade-in and fade-out) to a signal over a given duration.

    Args:
        signal (numpy.ndarray): Input audio signal (1D NumPy array).
        sample_rate (int): Sampling rate of the audio signal.
        ramp_duration_ms (float): Duration of the ramp in milliseconds (default: 5ms).

    Returns:
        numpy.ndarray: Signal with applied linear ramp.
    """
    num_samples = len(signal)
    ramp_samples = int((ramp_duration_ms / 1000) * sample_rate)  # Convert ms to samples

    if ramp_samples * 2 >= num_samples:
        raise ValueError("Ramp duration too long for the signal length!")

    # Create fade-in and fade-out ramps
    fade_in = np.linspace(0, 1, ramp_samples)
    fade_out = np.linspace(1, 0, ramp_samples)

    # Apply ramps to the signal
    signal[:ramp_samples] *= fade_in  # Apply fade-in
    signal[-ramp_samples:] *= fade_out  # Apply fade-out

    return signal


def low_freq_ratio(cochleagram, num_low_channels=5):
    """
    Compute the ratio of low-frequency energy to total energy.
    
    cochleagram: (channels, time) numpy array representing the cochleagram.
    num_low_channels: Number of lowest frequency channels to consider.
    
    Returns:
        ratio: Energy in the lowest `num_low_channels` divided by total energy.
    """
    low_energy = np.mean(cochleagram[:num_low_channels])  # Avg energy in first few channels
    total_energy = np.mean(cochleagram)  # Avg energy across all channels
    
    return low_energy / (total_energy + 1e-6)  # Avoid division by zero

def compute_low_freq_auc_ratio(cochleagram, num_low_freq_channels=5):
    """
    Computes the ratio of the area under the curve (AUC) for the first few 
    frequency channels to the AUC across all frequency channels.

    Args:
        cochleagram (torch.Tensor): Cochleagram matrix of shape (freq_channels, time_frames).
        num_low_freq_channels (int): Number of low-frequency channels to consider.

    Returns:
        float: Ratio of low-frequency AUC to total AUC.
    """
    # Compute AUC by summing across time frames
    auc_per_channel = cochleagram.sum(dim=1)  # Sum over time

    # Compute total AUC across all frequency channels
    total_auc = auc_per_channel.sum()

    # Compute AUC for the lowest frequency channels
    low_freq_auc = auc_per_channel[:num_low_freq_channels].sum()

    # Compute the ratio
    ratio = low_freq_auc / (total_auc + 1e-8)  # Small epsilon to avoid division by zero

    return ratio.item()


def compute_low_freq_auc_ratio_np(cochleagram, num_low_freq_channels=5):
    """
    Computes the ratio of the area under the curve (AUC) for the first few 
    frequency channels to the AUC across all frequency channels using NumPy.

    Args:
        cochleagram (np.ndarray): Cochleagram matrix of shape (freq_channels, time_frames).
        num_low_freq_channels (int): Number of low-frequency channels to consider.

    Returns:
        float: Ratio of low-frequency AUC to total AUC.
    """
    auc_per_channel = cochleagram.sum(axis=1)  # Sum over time
    total_auc = auc_per_channel.sum()
    low_freq_auc = auc_per_channel[:num_low_freq_channels].sum()
    ratio = low_freq_auc / (total_auc + 1e-8)  # Small epsilon to avoid division by zero
    return ratio

threshold = 0.75
# Example usage



In [ ]:
def select_best_pair_l2(time_avg_norms, segment_times):
    """
    Selects the pair of unique audio clips that minimizes cochleagram correlation 
    while maximizing time-averaged cochleagram correlation.
    
    Args:
        cochleagram_corrs (torch.Tensor): 4x4 correlation matrix of cochleagrams.
        time_avg_corrs (torch.Tensor): 4x4 correlation matrix of time-averaged cochleagrams.

    Returns:
        tuple: The selected pair of clip indices.
    """
    num_clips = time_avg_norms.shape[0]

    # Ensure matrices are converted to NumPy for easier indexing
    time_avg_norms = time_avg_norms.cpu().numpy()

    # Get all unique pairs of indices (excluding diagonal and duplicate pairs)
    pairs = list(itertools.combinations(range(num_clips), 2))

    best_pair = None
    best_score = float('inf')  # We want to maximize this score

    valid_time_norms = []

    segment_duration = 0.5
    best_time_norm = 0
    best_coch_corr = 1
    epsilon = 1e-5

    scores = []
    stim_scores = []
    
    for i, j in pairs:

        if abs(segment_times[i] - segment_times[j]) >= segment_duration:
            time_norm = time_avg_norms[i, j]     # lower is better
            
            valid_time_norms.append(time_avg_norms[i,j])
    
            # Define a scoring function: maximize time correlation, minimize cochleagram correlation
            stim_scores.append(time_norm)

            # minimize the l2 distance
            if time_norm < best_score:
                best_score = time_norm
                best_time_norm = time_norm
                best_pair = (i, j)

        scores.append(stim_scores)

    return best_pair, best_score,  time_avg_norms, valid_time_norms, scores

def select_best_pair_ratio(time_avg_norms, cochleagram_norms, segment_times):
    """
    Selects the pair of unique audio clips that maximizes the ratio of 
    time-averaged cochleagram norms to full cochleagram norms.

    Args:
        time_avg_norms (torch.Tensor): Matrix of L2 norms for time-averaged cochleagrams.
        cochleagram_norms (torch.Tensor): Matrix of L2 norms for full cochleagrams.
        segment_times (list or torch.Tensor): List of segment times corresponding to each clip.

    Returns:
        tuple: The selected pair of clip indices.
    """
    num_clips = time_avg_norms.shape[0]

    # Convert to NumPy for easier indexing
    time_avg_norms = time_avg_norms.cpu().numpy()
    cochleagram_norms = cochleagram_norms.cpu().numpy()

    # Get all unique pairs of indices (excluding diagonal and duplicate pairs)
    pairs = list(itertools.combinations(range(num_clips), 2))

    best_pair = None
    best_ratio = 0  # We want to maximize this ratio

    segment_duration = 0.5
    valid_ratios = []
    scores = []
    
    for i, j in pairs:
        if abs(segment_times[i] - segment_times[j]) >= segment_duration:
            # Compute the ratio of norms
            ratio = time_avg_norms[i, j] / (cochleagram_norms[i, j] + 1e-5)  # Avoid division by zero
            
            valid_ratios.append(ratio)
            scores.append(ratio)

            # Maximize this ratio
            if ratio > best_ratio:
                best_ratio = ratio
                best_pair = (i, j)

    return best_pair, best_ratio, valid_ratios, scores


# Define output file paths
csv_filename = save_dir + "best_sounds.csv"
npy_filename = save_dir + "cochleagram_data.npy"

best_sounds = []

target_sr = 22050*2
segment_duration = 0.5  # seconds
offset = 0.1
clip_duration = 10.0  # seconds
segment_starts = np.arange(0, clip_duration - segment_duration, offset)
num_samples_per_segment = int(segment_duration * target_sr)
segment_times = []

for start in segment_starts:
    start_sample = int(start * target_sr)
    end_sample = start_sample + num_samples_per_segment
    if end_sample <= clip_duration * target_sr:
        #segments.append(rms_normalize(wav_data[start_sample:end_sample]))
        segment_times.append(start)  # Adjust to clip's time in full sound

if os.path.exists(csv_filename) and os.path.exists(npy_filename):
    print("Loading existing data...")

    # Load cochleagram data
    all_cochleagrams = np.load(npy_filename, allow_pickle=True)

    # Read CSV file
    with open(csv_filename, mode='r') as f:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            curr_sound = row["curr_sound"]
            best_ratio = float(row["best_ratio"])
            best_pair = ast.literal_eval(row["best_pair"])  # Convert string to tuple
            onset_1 = float(row["segment_1_onset"])
            onset_2 = float(row["segment_2_onset"])

            coch_data = all_cochleagrams[i]
            segment_1 = rms_normalize(librosa.load(base_dir + curr_sound, sr=target_sr)[0][
                int(onset_1 * target_sr):int((onset_1 + 0.5) * target_sr)
            ])
            segment_2 = rms_normalize(librosa.load(base_dir + curr_sound, sr=target_sr)[0][
                int(onset_2 * target_sr):int((onset_2 + 0.5) * target_sr)
            ])

            segment_1 = apply_linear_ramp(segment_1, target_sr)
            segment_2 = apply_linear_ramp(segment_2, target_sr)

            best_sounds.append((
                curr_sound,
                segment_1,
                segment_2,
                best_ratio,
                best_pair,
                coch_data["cochleagram_1"],
                coch_data["cochleagram_2"],
                coch_data["time_avg_cochleagram_1"],
                coch_data["time_avg_cochleagram_2"],
            ))
    print(f"Loaded {len(best_sounds)} best sounds from cache.")
else:
    print("No cache found. Running full processing pipeline...")
    # Your entire existing code goes here, including the main loop over `fps`

    pairs_ = []
    best_time_corrs_ = []
    best_corrs_ = []
    corrs_ = []
    time_corrs = []
    best_sounds = []
    ss_scores = []
    target_stationarity = 0
    
    target_sr = 22050*2
    
    k = 0
    
    best_pairs_dict = defaultdict(list)
    
    # Define output file paths
    csv_filename = save_dir + "best_sounds.csv"
    npy_filename = save_dir + "cochleagram_data.npy"
    
    # Storage for all cochleagrams
    all_cochleagrams = []
    
    # Ensure CSV file has a header if it doesn't exist
    with open(csv_filename, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["curr_sound", "best_ratio", "best_pair", "segment_1_onset", "segment_2_onset"])
    
    for curr_sound in fps:
    
        # Compute cochleagram for each segment
        cochleagrams = []
        time_averaged_specs = []
    
        stationarity, times = ss_scores_to_texture[curr_sound], times_to_texture[curr_sound]
    
        # if you have no stationarity scores for this sound, then move on to the next
        if len(stationarity) <= 0:
            #print("skipped")
            continue
        
        audio_path = base_dir + curr_sound
        
        data, samplerate = librosa.load(audio_path)
        
        if samplerate != target_sr:
            data = librosa.resample(data, orig_sr=samplerate, target_sr=target_sr)
    
        data = rms_normalize(data)
        
        # Parameters
        sr = samplerate
        segment_duration = 0.5  # seconds
        clip_duration = 10.0  # seconds
        num_samples_per_segment = int(segment_duration * target_sr)
        
        #print(best_ss)
        min_mse = 1e9
        best_x = -1
        best_time = 0
        best_ss = 0
    
        best_x, best_time, best_ss = find_max_stationarity(times, stationarity, target_stationarity)
    
        #print("best ss", best_ss)
        ss_scores.append(best_ss)
        
        #sound = data[int(best_time*target_sr):int((best_time+2)*target_sr)]
        wav_data = data
        
        # Generate all possible 0.5s segments
        segment_starts = np.arange(0, clip_duration - segment_duration, offset)  # Shift every 0.1s
        segments = []
        segment_times = []
        
        for start in segment_starts:
            start_sample = int(start * sr)
            end_sample = start_sample + num_samples_per_segment
            if end_sample <= len(wav_data):
                segments.append(rms_normalize(wav_data[start_sample:end_sample]))
                segment_times.append(start)  # Adjust to clip's time in full sound
        
        for segment in segments:
            segment_torch = torch.from_numpy(segment).unsqueeze(0).unsqueeze(0)
            S = model(segment_torch.to("cuda"))
        
            cochleagrams.append(S)
        
        #print(len(segments), len(cochleagrams))
        
        # calculating correlations between all cochleagrams
        cochleagram_tensor = torch.stack(cochleagrams).squeeze()
        # Reshape each cochleagram into a single vector per sound
        flattened_full_coch = cochleagram_tensor.reshape(cochleagram_tensor.shape[0], -1)
        
        # Compute pairwise L2 norms
        full_coch_l2 = torch.cdist(flattened_full_coch, flattened_full_coch, p=2)
    
        
        time_averaged_cochleagram = cochleagram_tensor.mean(dim=2)  
    
        # Reshape for pairwise distance computation
        flattened = time_averaged_cochleagram.reshape(time_averaged_cochleagram.shape[0], -1)
        
        # Compute pairwise L2 norms
        time_averaged_coch_l2 = torch.cdist(flattened, flattened, p=2)
    
        #print(time_averaged_coch_l2.shape)
    
        # time_averaged_coch_corr = torch.corrcoef(time_averaged_cochleagram.reshape(time_averaged_cochleagram.shape[0], -1))
    
        # print(time_averaged_coch_corr.shape)
    
        #print(time_averaged_coch_l2, segment_times)
    
        best_pair, best_ratio, valid_ratios, scores = select_best_pair_ratio(time_averaged_coch_l2, full_coch_l2, segment_times)
    
    
        print(best_pair)
        # Extract the two most different segments
        segment_1 = segments[best_pair[0]]
        segment_2 = segments[best_pair[1]]
    
    
        ratio = compute_low_freq_auc_ratio_np(np.flipud(cochleagram_tensor.cpu().numpy()[best_pair[0]]))
    
        if ratio > threshold:  # Define a reasonable threshold
            continue
            print("Filtered out: Too much low-frequency energy")
        else:
            print(ratio)
            #plt.scatter(np.arange(40), best_sounds[100][-1][::-1])
            fig, axes = plt.subplots(nrows=1, ncols=1, figsize=(10, 20), sharex=True, sharey=True)
            #orch.flip(cochleagram, dims=[0])
            axes.imshow(np.flipud(cochleagram_tensor.cpu().numpy()[0]), aspect="auto", origin="lower", cmap="inferno", vmin=0, vmax=0.32)
            fig.show()
            #input()
    
        ratio = compute_low_freq_auc_ratio_np(np.flipud(cochleagram_tensor.cpu().numpy()[best_pair[1]]))
    
        if ratio > threshold:  # Define a reasonable threshold
            continue
            print("Filtered out: Too much low-frequency energy")
        # #print(best_pair[0], best_pair[1])
        
        #print(f"Most different segments: {segment_times[best_pair[0]]:.2f}s and {segment_times[best_pair[1]]:.2f}s")
        #print(f"Most different segments: {segment_times[best_pair[0]]}s and {segment_times[best_pair[1]]}s with time avg. coch norm {best_score} ")
        
        norm_s1 = rms_normalize(segment_1)
        norm_s2 = rms_normalize(segment_2)
    
        norm_s1 = apply_linear_ramp(norm_s1, target_sr)
        norm_s2 = apply_linear_ramp(norm_s2, target_sr)
    
        # best_sounds.append((curr_sound, norm_s1, 
        #                     norm_s2, best_ratio, 
        #                     best_pair, 
        #                     cochleagram_tensor.cpu().numpy()[best_pair[0]],
        #                     cochleagram_tensor.cpu().numpy()[best_pair[1]],
        #                     time_averaged_cochleagram.cpu().numpy()[best_pair[0]],
        #                     time_averaged_cochleagram.cpu().numpy()[best_pair[1]]))
    
        # Save metadata to CSV
        with open(csv_filename, mode='a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([curr_sound, best_ratio, best_pair, segment_times[best_pair[0]], segment_times[best_pair[1]]])
    
        # Append to the list instead of overwriting
        cochleagram_data = {
            "curr_sound": curr_sound,
            "cochleagram_1": cochleagram_tensor.cpu().numpy()[best_pair[0]],
            "cochleagram_2": cochleagram_tensor.cpu().numpy()[best_pair[1]],
            "time_avg_cochleagram_1": time_averaged_cochleagram.cpu().numpy()[best_pair[0]],
            "time_avg_cochleagram_2": time_averaged_cochleagram.cpu().numpy()[best_pair[1]],
        }
        all_cochleagrams.append(cochleagram_data)
        
        # display(ipd.Audio(norm_s1, rate=target_sr))
        # display(ipd.Audio(norm_s2, rate=target_sr))
    
        #input()
        
        clear_output(wait=False)
    
        # Save all cochleagrams at the end
        np.save(npy_filename, all_cochleagrams)


In [ ]:
def equal_width_bar(to_plot, data=None, num_bins=5, label=""):
    if data == None:
        bin_edges = np.linspace(min(to_plot), max(to_plot), num_bins + 1)  # Evenly spaced bins
    else:
        bin_edges = np.linspace(min(data), max(data), num_bins + 1)  # Evenly spaced bins
    
    # Compute histogram using numpy
    counts, edges = np.histogram(to_plot, bins=bin_edges, density=True)

    # Plot histogram manually with specified bin widths
    plt.bar(edges[:-1], counts, width=np.diff(edges), edgecolor='black', alpha=0.7, label=label)

In [ ]:
import shutil

# Assuming best_sounds is a list of tuples (curr_sound, norm_s1, norm_s2, best_score, best_pair)
# Sorting by best_score| (ascending order, since smaller is better)
best_sounds_sorted = sorted(best_sounds, key=lambda x: x[3])

top_10_sounds    = best_sounds_sorted[:10]
bottom_10_sounds = best_sounds_sorted[-10:]


all_norms = [x[3] for x in best_sounds_sorted]

worst_norms = [x[3] for x in bottom_10_sounds]
best_norms  = [x[3] for x in top_10_sounds]

#equal_width_bar(diff_times)

# plt.hist(all_norms, bins=len(all_norms), alpha=0.5)
# plt.hist(worst_norms, bins=len(all_norms), color='r', alpha=0.5, label='worst')
# plt.hist(best_norms, bins=len(best_norms), color='g', alpha=0.5, label='best')

num_bins = 50
equal_width_bar(all_norms, num_bins=num_bins, label="All")
equal_width_bar(worst_norms, data=all_norms, num_bins=num_bins, label="Worst")
equal_width_bar(best_norms, data=all_norms, num_bins=num_bins, label="Best")
plt.legend()


plt.title(f"Minimized Segment Ratio of Norms across all sounds with s.s < {avg_ss_score:.2f}")
# Extracting top 10 (smallest scores) and bottom 10 (largest scores)

# Define output directories for saving the wav files
output_dir = save_dir
top_dir = os.path.join(output_dir, "top_10")
bottom_dir = os.path.join(output_dir, "bottom_10")

# Create directories if they don't exist
os.makedirs(top_dir, exist_ok=True)
os.makedirs(bottom_dir, exist_ok=True)

# Function to save wav files
def save_wavs(sound_list, directory):
    for i, (curr_sound, norm_s1, norm_s2, best_score, best_pair, _, _, _, _) in enumerate(sound_list):
        
        audio_path = base_dir + curr_sound
        
        
        sound_name = curr_sound.split("/")[-1].split(".")[0]

        wavfile.write(os.path.join(directory, f"{sound_name}_1.wav"), target_sr, norm_s1.astype(np.float32))
        wavfile.write(os.path.join(directory, f"{sound_name}_2.wav"), target_sr, norm_s2.astype(np.float32))
        

#Save top 10 sounds
save_wavs(top_10_sounds, top_dir)

#Save bottom 10 sounds
save_wavs(bottom_10_sounds, bottom_dir)

print("Top 10 and bottom 10 sounds saved successfully!")

In [ ]:
best_sounds[1][-1]

In [ ]:
diff_times = []
for sound in best_sounds_sorted:
    times = sound[4]
    diff_times.append(abs(segment_times[times[0]] - segment_times[times[1]]))

equal_width_bar(diff_times)

print(bottom_dir)

In [ ]:

fig, axes = plt.subplots(nrows=10, ncols=2, figsize=(10, 20), sharex=True, sharey=True)

for i in range(10):
    # Extract filenames
    bottom_filename = bottom_10_sounds[i][0].split("/")[-1]
    top_filename = top_10_sounds[i][0].split("/")[-1]


    # Plot bottom 10 sounds
    axes[i, 0].plot(np.arange(40), bottom_10_sounds[i][-1][::-1], label="Sound 1", linestyle="-", marker="o")
    axes[i, 0].plot(np.arange(40), bottom_10_sounds[i][-2][::-1], label="Sound 2", linestyle="--", marker="s")
    axes[i, 0].set_ylabel("Averaged Response")
    axes[i, 0].grid(True)
    norm_diff_bottom = bottom_10_sounds[i][3]  # Extract norm of difference
    axes[i, 0].text(30, np.max(bottom_10_sounds[i][-1]) + 0.01, f"Ratio: {norm_diff_bottom:.2f}", fontsize=10, color="red")

    # Add filename annotation at the top of each subplot
    axes[i, 0].annotate(f"{bottom_filename}", xy=(0.5, 1.05), xycoords="axes fraction", 
                        ha="center", fontsize=9, fontweight="bold")

    # Plot top 10 sounds
    axes[i, 1].plot(np.arange(40), top_10_sounds[i][-1][::-1], label="Sound 1", linestyle="-", marker="o")
    axes[i, 1].plot(np.arange(40), top_10_sounds[i][-2][::-1], label="Sound 2", linestyle="--", marker="s")
    axes[i, 1].grid(True)
    norm_diff_top = top_10_sounds[i][3]  # Extract norm of difference
    axes[i, 1].text(30, np.max(top_10_sounds[i][-1]) + 0.01, f"Ratio: {norm_diff_top:.2f}", fontsize=10, color="red")

    # Add filename annotation at the top of each subplot
    axes[i, 1].annotate(f"{top_filename}", xy=(0.5, 1.05), xycoords="axes fraction", 
                        ha="center", fontsize=9, fontweight="bold")



# Add a main figure title
fig.suptitle("Time-Averaged Cochleagrams: Best Pairs Selected Using Ratio of Norm of the Difference", 
             fontsize=14, fontweight="bold", y=1.02)

# Move column titles higher
fig.text(0.25, 0.98, "Bottom 10 Sounds", fontsize=12, fontweight="bold", ha="center")
fig.text(0.75, 0.98, "Top 10 Sounds", fontsize=12, fontweight="bold", ha="center")

# Common x-label and legend
axes[-1, 0].set_xlabel("Cochlear Channel")
axes[-1, 1].set_xlabel("Cochlear Channel")
axes[0, 0].legend()
plt.tight_layout(rect=[0, 0, 1, 0.98])  # Adjust layout to fit title and avoid too much space

plt.savefig(save_dir + "summary.png")
plt.clf()

In [ ]:
# y_ticks = np.arange(0, 40+ (num_channels//10), step=max(1, num_channels//10))
# y_ticks

In [ ]:
for i in range(10):
    sound1_cochleagram = bottom_10_sounds[i][-4]  # Cochleagram for Sound 1
    sound2_cochleagram = bottom_10_sounds[i][-3]  # Cochleagram for Sound 2
    num_channels = sound1_cochleagram.shape[0]  # Get number of cochlear channels

    y_ticks = np.arange(0, num_channels, step=max(1, num_channels // 10))  # Ensure spacing
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Extract filename for labeling
    filename = top_10_sounds[i][0].split("/")[-1]

    # Plot Sound 1 cochleagram
    im1 = axes[0].imshow(sound1_cochleagram, aspect="auto", origin="lower", cmap="inferno", vmin=0, vmax=0.32)
    axes[0].set_title(f"Segment 1 Cochleagram\n{filename.split('.')[0]} ({segment_times[top_10_sounds[i][4][0]]:.2f}s)", fontsize=12)
    axes[0].set_xlabel("Time (frames)")
    axes[0].set_ylabel("Cochlear Channel")
    axes[0].set_yticks(y_ticks)
    axes[0].invert_yaxis()  # Flip the axis correctly
    fig.colorbar(im1, ax=axes[0])

    # Plot Sound 2 cochleagram
    im2 = axes[1].imshow(sound2_cochleagram, aspect="auto", origin="lower", cmap="inferno", vmin=0, vmax=0.32)
    axes[1].set_title(f"Segment 2 Cochleagram\n{filename.split('.')[0]} ({segment_times[top_10_sounds[i][4][1]]:.2f}s)", fontsize=12)
    axes[1].set_xlabel("Time (frames)")
    axes[1].set_yticks(y_ticks)
    axes[1].invert_yaxis()  # Flip the axis correctly
    fig.colorbar(im2, ax=axes[1])

    # Adjust layout
    plt.suptitle("Cochleagrams for a Selected Pair (from NORM) from BOTTOM 10 Sounds", fontsize=14, fontweight="bold")
    plt.tight_layout()

    # Save figure
    plt.savefig(bottom_dir + f"/{filename.split('.')[0]}-cochleagrams.png")
    plt.clf()

In [ ]:
import numpy as np
for i in range(10):
    sound1_cochleagram = top_10_sounds[i][-4]  # Cochleagram for Sound 1
    sound2_cochleagram = top_10_sounds[i][-3]  # Cochleagram for Sound 2
    num_channels = sound1_cochleagram.shape[0]  # Get number of cochlear channels

    y_ticks = np.arange(0, num_channels, step=max(1, num_channels // 10))  # Ensure spacing
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Extract filename for labeling
    filename = top_10_sounds[i][0].split("/")[-1]

    # Plot Sound 1 cochleagram
    im1 = axes[0].imshow(sound1_cochleagram, aspect="auto", origin="lower", cmap="inferno", vmin=0, vmax=0.32)
    axes[0].set_title(f"Segment 1 Cochleagram\n{filename.split('.')[0]} ({segment_times[top_10_sounds[i][4][0]]:.2f}s)", fontsize=12)
    axes[0].set_xlabel("Time (frames)")
    axes[0].set_ylabel("Cochlear Channel")
    axes[0].set_yticks(y_ticks)
    axes[0].invert_yaxis()  # Flip the axis correctly
    fig.colorbar(im1, ax=axes[0])

    # Plot Sound 2 cochleagram
    im2 = axes[1].imshow(sound2_cochleagram, aspect="auto", origin="lower", cmap="inferno", vmin=0, vmax=0.32)
    axes[1].set_title(f"Segment 2 Cochleagram\n{filename.split('.')[0]} ({segment_times[top_10_sounds[i][4][1]]:.2f}s)", fontsize=12)
    axes[1].set_xlabel("Time (frames)")
    axes[1].set_yticks(y_ticks)
    axes[1].invert_yaxis()  # Flip the axis correctly
    fig.colorbar(im2, ax=axes[1])

    # Adjust layout
    plt.suptitle("Cochleagrams for a Selected Pair (from NORM) from TOP 10 Sounds", fontsize=14, fontweight="bold")
    plt.tight_layout()

    # Save figure
    plt.savefig(top_dir + f"/{filename.split('.')[0]}-cochleagrams.png")
    plt.clf()

In [ ]:
def select_best_pair_corr(time_avg_corrs, segment_times):
    """
    Selects the pair of unique audio clips that minimizes cochleagram correlation 
    while maximizing time-averaged cochleagram correlation.
    
    Args:
        cochleagram_corrs (torch.Tensor): 4x4 correlation matrix of cochleagrams.
        time_avg_corrs (torch.Tensor): 4x4 correlation matrix of time-averaged cochleagrams.

    Returns:
        tuple: The selected pair of clip indices.
    """
    num_clips = time_avg_corrs.shape[0]

    # Ensure matrices are converted to NumPy for easier indexing
    time_avg_corrs = time_avg_corrs.cpu().numpy()
    
    # Get all unique pairs of indices (excluding diagonal and duplicate pairs)
    pairs = list(itertools.combinations(range(num_clips), 2))

    best_pair = None
    best_score = -float('inf')  # We want to maximize this score

    valid_corrs = []
    valid_time_corrs = []

    segment_duration = 0.5
    best_time_corr = 0
    best_coch_corr = 1
    epsilon = 1e-5

    scores = []
    
    for i, j in pairs:

        if abs(segment_times[i] - segment_times[j]) >= segment_duration:
            time_corr = time_avg_corrs[i, j]     # Higher is better

            valid_time_corrs.append(time_corr)
            # Define a scoring function: maximize time correlation, minimize cochleagram correlation
            score = time_corr  # Simple difference; can be weighted differently if needed
            scores.append(score)
    
            if score > best_score:
                best_score = score
                best_time_corr = time_avg_corrs[i,j] 
                best_pair = (i, j)

    return best_pair, best_time_corr, valid_time_corrs, scores, best_score

pairs_ = []
best_time_corrs_ = []
best_corrs_ = []
corrs_ = []
time_corrs = []
best_sounds = []
ss_scores = []
target_stationarity = 0

target_sr = 22050*2

k = 0

best_pairs_dict = defaultdict(list)



for curr_sound in fps:

    # Compute cochleagram for each segment
    cochleagrams = []
    time_averaged_specs = []

    stationarity, times = ss_scores_to_texture[curr_sound], times_to_texture[curr_sound]

    # if you have no stationarity scores for this sound, then move on to the next
    if len(stationarity) <= 0:
        #print("skipped")
        continue
    
    audio_path = base_dir + curr_sound
    
    data, samplerate = librosa.load(audio_path)
    
    if samplerate != target_sr:
        data = librosa.resample(data, orig_sr=samplerate, target_sr=target_sr)

    data = rms_normalize(data)
    
    # Parameters
    sr = samplerate
    segment_duration = 0.5  # seconds
    clip_duration = 10.0  # seconds
    num_samples_per_segment = int(segment_duration * target_sr)
    
    #print(best_ss)
    min_mse = 1e9
    best_x = -1
    best_time = 0
    best_ss = 0

    best_x, best_time, best_ss = find_max_stationarity(times, stationarity, target_stationarity)

    #print("best ss", best_ss)
    ss_scores.append(best_ss)
    
    sound = data[int(best_time*target_sr):int((best_time+2)*target_sr)]
    wav_data = sound
    
    
    # Generate all possible 0.5s segments
    segment_starts = np.arange(0, clip_duration - segment_duration, offset)  # Shift every 0.1s
    segments = []
    segment_times = []
    
    for start in segment_starts:
        start_sample = int(start * sr)
        end_sample = start_sample + num_samples_per_segment
        if end_sample <= len(wav_data):
            segments.append(rms_normalize(wav_data[start_sample:end_sample]))
            segment_times.append(start)  # Adjust to clip's time in full sound
    
    for segment in segments:
        segment_torch = torch.from_numpy(segment).unsqueeze(0).unsqueeze(0)
        S = model(segment_torch.to("cuda"))
    
        cochleagrams.append(S)

    cochleagram_tensor = torch.stack(cochleagrams).squeeze()
    time_averaged_cochleagram = cochleagram_tensor.mean(dim=2)  

    # Reshape for pairwise distance computation
    flattened = time_averaged_cochleagram.reshape(time_averaged_cochleagram.shape[0], -1)
    

    time_averaged_coch_corr = torch.corrcoef(time_averaged_cochleagram.reshape(time_averaged_cochleagram.shape[0], -1))

    # print(time_averaged_coch_corr.shape)

    #print(time_averaged_coch_corr, segment_times)

    best_pair, best_time_corr, valid_time_corrs, scores, best_score = select_best_pair_corr(time_averaged_coch_corr, segment_times)


    #print(best_pair)
    # Extract the two most different segments
    segment_1 = segments[best_pair[0]]
    segment_2 = segments[best_pair[1]]

    
    # #print(best_pair[0], best_pair[1])
    
    #print(f"Most different segments: {segment_times[best_pair[0]]:.2f}s and {segment_times[best_pair[1]]:.2f}s")
    print(f"Most different segments: {segment_times[best_pair[0]]}s and {segment_times[best_pair[1]]}s with time avg. coch norm {best_score} ")
    
    norm_s1 = rms_normalize(segment_1)
    norm_s2 = rms_normalize(segment_2)

    norm_s1 = apply_linear_ramp(norm_s1, target_sr)
    norm_s2 = apply_linear_ramp(norm_s2, target_sr)

    best_sounds.append((curr_sound, norm_s1, norm_s2, best_score, best_pair,
                        time_averaged_cochleagram.cpu().numpy()[best_pair[0]],
                        time_averaged_cochleagram.cpu().numpy()[best_pair[1]]))
    
    # display(ipd.Audio(norm_s1, rate=target_sr))
    # display(ipd.Audio(norm_s2, rate=target_sr))

    #input()
    
    clear_output(wait=False)


In [ ]:
import shutil
save_dir = f"/om2/user/bjmedina/data/texture_pairs/cochleagram_corrs-max_stationarity-lessthan{avg_ss_score:.2f}-offset{offset}/"
#save_dir = f"/om2/user/bjmedina/data/texture_pairs/cochleagram_corrs-max_stationarity-lessthan{avg_ss_score:.2f}-offset{offset}/"

# Assuming best_sounds is a list of tuples (curr_sound, norm_s1, norm_s2, best_score, best_pair)
# Sorting by best_score (ascending order, since smaller is better)
best_sounds_sorted = sorted(best_sounds, key=lambda x: x[3])
top_10_sounds    = best_sounds_sorted[-10:]
bottom_10_sounds = best_sounds_sorted[:10]


all_norms = [x[3] for x in best_sounds_sorted]

worst_norms = [x[3] for x in bottom_10_sounds]
best_norms  = [x[3] for x in top_10_sounds]

plt.hist(all_norms, bins=len(all_norms), alpha=0.5)
plt.hist(worst_norms, bins=5, color='r', alpha=0.5, label='worst')
plt.hist(best_norms, bins=5, color='g', alpha=0.5, label='best')


plt.title(f"Minimized Segment Norms across all sounds with  s.s < {avg_ss_score:.2f}")
# Extracting top 10 (smallest scores) and bottom 10 (largest scores)

# Define output directories for saving the wav files
output_dir = save_dir
top_dir = os.path.join(output_dir, "top_10")
bottom_dir = os.path.join(output_dir, "bottom_10")

# Create directories if they don't exist
os.makedirs(top_dir, exist_ok=True)
os.makedirs(bottom_dir, exist_ok=True)

# Function to save wav files
def save_wavs(sound_list, directory):
    for i, (curr_sound, norm_s1, norm_s2, best_score, best_pair, _, _) in enumerate(sound_list):
        wavfile.write(os.path.join(directory, f"sound_{i+1}_s1.wav"), target_sr, norm_s1.astype(np.float32))
        wavfile.write(os.path.join(directory, f"sound_{i+1}_s2.wav"), target_sr, norm_s2.astype(np.float32))
        

# # Save top 10 sounds
# save_wavs(top_10_sounds, top_dir)

# # Save bottom 10 sounds
# save_wavs(bottom_10_sounds, bottom_dir)

print("Top 10 and bottom 10 sounds saved successfully!")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(nrows=10, ncols=2, figsize=(10, 20), sharex=True, sharey=True)

for i in range(10):
    # Extract filenames
    bottom_filename = bottom_10_sounds[i][0].split("/")[-1]
    top_filename = top_10_sounds[i][0].split("/")[-1]
    
    # Plot bottom 10 sounds (based on correlation)
    axes[i, 0].plot(np.arange(40), bottom_10_sounds[i][-1], label="Sound 1", linestyle="-", marker="o")
    axes[i, 0].plot(np.arange(40), bottom_10_sounds[i][-2], label="Sound 2", linestyle="--", marker="s")
    axes[i, 0].set_ylabel("Averaged Response")
    axes[i, 0].grid(True)
    corr_bottom = bottom_10_sounds[i][3]  # Extract correlation value
    axes[i, 0].text(30, np.max(bottom_10_sounds[i][-1]) + 0.01, f"Corr: {corr_bottom:.2f}", fontsize=10, color="blue")

    # Add filename annotation at the top of each subplot
    axes[i, 0].annotate(f"{bottom_filename}", xy=(0.5, 1.05), xycoords="axes fraction", 
                        ha="center", fontsize=9, fontweight="bold")

    # Plot top 10 sounds (based on correlation)
    axes[i, 1].plot(np.arange(40), top_10_sounds[i][-1], label="Sound 1", linestyle="-", marker="o")
    axes[i, 1].plot(np.arange(40), top_10_sounds[i][-2], label="Sound 2", linestyle="--", marker="s")
    axes[i, 1].grid(True)
    corr_top = top_10_sounds[i][3]  # Extract correlation value
    axes[i, 1].text(30, np.max(top_10_sounds[i][-1]) + 0.01, f"Corr: {corr_top:.2f}", fontsize=10, color="blue")

    # Add filename annotation at the top of each subplot
    axes[i, 1].annotate(f"{top_filename}", xy=(0.5, 1.05), xycoords="axes fraction", 
                        ha="center", fontsize=9, fontweight="bold")

# Add a main figure title
fig.suptitle("Time-Averaged Cochleagrams: Best Pairs Selected Using Correlation", 
             fontsize=14, fontweight="bold", y=1.02)

# Move column titles higher
fig.text(0.25, 0.98, "Bottom 10 Sounds", fontsize=12, fontweight="bold", ha="center")
fig.text(0.75, 0.98, "Top 10 Sounds", fontsize=12, fontweight="bold", ha="center")

# Common x-label and legend
axes[-1, 0].set_xlabel("Cochlear Channel")
axes[-1, 1].set_xlabel("Cochlear Channel")
axes[0, 0].legend()
plt.tight_layout(rect=[0, 0, 1, 0.98])  # Adjust layout to fit title and avoid too much space

plt.show()

In [ ]:
plt.hist(ss_scores, bins=len(ss_scores))
np.sum(np.array(ss_scores)>-0.4)

In [ ]:
# plt.hist(time_corrs, bins=1000, alpha=0.5, density=True)
# plt.hist(best_time_corrs_, bins=1000, alpha=0.5, density=True
print("Average correlation between time-averaged cochleagrams", np.mean(time_corrs))

In [ ]:
# plt.hist(corrs_, bins=1000, alpha=0.5, density=True)
# plt.hist(best_corrs_, bins=1000, alpha=0.5, density=True
print("Average correlation between cochleagrams", np.mean(corrs_))

In [ ]:
print(len(best_time_corrs_))

In [ ]:
stat_cutoff = -0.499
save_dir_filtered = f"/om2/user/bjmedina/data/texture_pairs/cochleagram_corrs-max_stationarity_filtered_greaterthan{stat_cutoff}_offset-{offset}/"

In [ ]:
# Get indices where A > a and B < b

mean_corr = np.mean(corrs_)
std_corr  = 0#np.std(corrs_)
mean_time_corr = np.mean(time_corrs)
std_time_corr  = 0#np.std(time_corrs)
indices = [i for i in range(len(best_corrs_)) if (best_corrs_[i] <= mean_corr and best_time_corrs_[i] >= mean_time_corr and ss_scores[i] >= stat_cutoff)]

print(len(indices))

In [ ]:
filtered_best = [best_sounds[best_idx] for best_idx in indices]

In [ ]:
best_sounds[0][1]

In [ ]:
# norm_s1 = rms_normalize(segment_1)
# norm_s2 = rms_normalize(segment_2)
chosen_idx = 14
# # display(ipd.Audio(norm_s1, rate=samplerate))
# # display(ipd.Audio(norm_s2, rate=samplerate))
# display(ipd.Audio(filtered_best[chosen_idx][1], rate=target_sr))
# display(ipd.Audio(filtered_best[chosen_idx][2], rate=target_sr))

for ID, pair in enumerate(filtered_best):
    
    segment_1 = pair[1]
    segment_2 = pair[2]    
    #print(f"Most different segments: {segment_times[best_pair[0]]:.2f}s and {segment_times[best_pair[1]]:.2f}s")
    # #print(f"Most different segments: {segment_times[best_pair[0]]}s and {segment_times[best_pair[1]]}s with distance {max_distance}")
    #display(ipd.Audio(norm_s1, rate=target_sr))
    #display(ipd.Audio(norm_s2, rate=target_sr))

    if not os.path.exists(save_dir_filtered):
        os.makedirs(save_dir_filtered)
    
    #print(save_dir+f"{ID}_0.wav")

    wavfile.write(save_dir_filtered+f"mem{ID}_0.wav", target_sr, segment_1.astype(np.float32))
    wavfile.write(save_dir_filtered+f"mem{ID}_1.wav", target_sr, segment_2.astype(np.float32))


In [ ]:
offset 

In [ ]:
`pairs_ = []
best_time_corrs_ = []
best_corrs_ = []
corrs_ = []
time_corrs = []
best_sounds = []
ss_scores = []
target_stationarity = 0

target_sr = 22050*2

k = 0

best_pairs_dict = defaultdict(list)

for curr_sound in fps:

    # Compute cochleagram for each segment
    cochleagrams = []
    time_averaged_specs = []

    stationarity, times = ss_scores_to_texture[curr_sound], times_to_texture[curr_sound]

    # if you have no stationarity scores for this sound, then move on to the next
    if len(stationarity) <= 0:
        #print("skipped")
        continue
    
    audio_path = base_dir + curr_sound
    
    data, samplerate = librosa.load(audio_path)
    
    if samplerate != target_sr:
        data = librosa.resample(data, orig_sr=samplerate, target_sr=target_sr)

    data = rms_normalize(data)
    
    # Parameters
    sr = samplerate
    segment_duration = 0.5  # seconds
    clip_duration = 10.0  # seconds
    num_samples_per_segment = int(segment_duration * target_sr)
    
    #print(best_ss)
    min_mse = 1e9
    best_x = -1
    best_time = 0
    best_ss = 0

    best_x, best_time, best_ss = find_max_stationarity(times, stationarity, target_stationarity)

    #print("best ss", best_ss)
    ss_scores.append(best_ss)
    
    sound = data[int(best_time*target_sr):int((best_time+2)*target_sr)]
    wav_data = sound
    
    
    # Generate all possible 0.5s segments
    segment_starts = np.arange(0, clip_duration - segment_duration, offset)  # Shift every 0.1s
    segments = []
    segment_times = []
    
    for start in segment_starts:
        start_sample = int(start * sr)
        end_sample = start_sample + num_samples_per_segment
        if end_sample <= len(wav_data):
            segments.append(rms_normalize(wav_data[start_sample:end_sample]))
            segment_times.append(start)  # Adjust to clip's time in full sound
    
    for segment in segments:
        segment_torch = torch.from_numpy(segment).unsqueeze(0).unsqueeze(0)
        S = model(segment_torch.to("cuda"))
    
        cochleagrams.append(S)
    
    #print(len(segments), len(cochleagrams))
    
    # calculating correlations between all cochleagrams
    cochleagram_tensor = torch.stack(cochleagrams).squeeze()
    coch_corr = torch.corrcoef(cochleagram_tensor.reshape(cochleagram_tensor.shape[0], -1))
    
    time_averaged_cochleagram = cochleagram_tensor.mean(dim=2)  

    # Reshape for pairwise distance computation
    flattened = time_averaged_cochleagram.reshape(time_averaged_cochleagram.shape[0], -1)
    
    # Compute pairwise L2 norms
    time_averaged_coch_l2 = torch.cdist(flattened, flattened, p=2)

    print(time_averaged_coch_l2.shape)

    ime_averaged_coch_corr = torch.corrcoef(time_averaged_cochleagram.reshape(time_averaged_cochleagram.shape[0], -1))

    # print(time_averaged_coch_corr.shape)

    print(time_averaged_coch_l2, segment_times)

    best_pair, best_time_corr, best_coch_corr, comparison_corrs, comparison_time_corrs, scores = select_best_pair_corr(coch_corr, time_averaged_coch_corr, segment_times)


    print(best_pair)
    # Extract the two most different segments
    segment_1 = segments[best_pair[0]]
    segment_2 = segments[best_pair[1]]

    
    # #print(best_pair[0], best_pair[1])
    
    #print(f"Most different segments: {segment_times[best_pair[0]]:.2f}s and {segment_times[best_pair[1]]:.2f}s")
    print(f"Most different segments: {segment_times[best_pair[0]]}s and {segment_times[best_pair[1]]}s with time avg. coch norm {best_score} ")
    
    norm_s1 = rms_normalize(segment_1)
    norm_s2 = rms_normalize(segment_2)

    norm_s1 = apply_linear_ramp(norm_s1, target_sr)
    norm_s2 = apply_linear_ramp(norm_s2, target_sr)

    best_sounds.append((curr_sound, norm_s1, norm_s2, best_score, best_pair ))
    
    # display(ipd.Audio(norm_s1, rate=target_sr))
    # display(ipd.Audio(norm_s2, rate=target_sr))

    #input()
    
    clear_output(wait=False)


    
    best_pair, best_time_corr, best_coch_corr, comparison_corrs, comparison_time_corrs, scores = select_best_pair(coch_corr, time_averaged_coch_corr, segment_times)
    best_time_corrs_.append(best_time_corr)
    best_corrs_.append(best_coch_corr)
    
    # saving all values (this is keeping the diagonals, maybe we don't want that)
    #print(f"Coch corrs: {comparison_corrs}")
    corrs_.extend(comparison_corrs)

    #print(f"Time avg. Coch corrs: {comparison_time_corrs}")
    time_corrs.extend(comparison_time_corrs)

    all_scores.extend(scores)


    to_save["best_time_corr"] = best_time_corr
    to_save["best_coch_corr"] = best_coch_corr
    to_save["clip_1_onset"]   = segment_times[best_pair[0]]
    to_save["clip_2_onset"]   = segment_times[best_pair[1]]
    to_save["best_pair"]      = best_pair
    to_save["segment_times"]  = segment_times
    to_save["all_coch_corrs"] = comparison_corrs
    to_save["all_time_coch_corrs"] = comparison_time_corrs
    to_save["scores"] = scores

    clip_info[curr_sound] = to_save
    
    
    
    # pairs_.append(best_pair)
    
    # # Extract the two most different segments
    # segment_1 = segments[best_pair[0]]
    # segment_2 = segments[best_pair[1]]

    # best_sounds.append((curr_sound, segment_1, segment_2))
    
    # # #print(best_pair[0], best_pair[1])
    
    # #print(f"Most different segments: {segment_times[best_pair[0]]:.2f}s and {segment_times[best_pair[1]]:.2f}s")
    # print(f"Most different segments: {segment_times[best_pair[0]]}s and {segment_times[best_pair[1]]}s with coch corr {best_coch_corr} and time avg corr {best_time_corr} ")
    
    # norm_s1 = rms_normalize(segment_1)
    # norm_s2 = rms_normalize(segment_2)
    
    # display(ipd.Audio(norm_s1, rate=target_sr))
    # display(ipd.Audio(norm_s2, rate=target_sr))
    #     os.makedirs(save_dir)
    
    # ID = curr_sound.split("/")[-1].split(".")[0]

    # #print(save_dir+f"{ID}_0.wav")

    # wavfile.write(save_dir+f"{ID}_0.wav", target_sr, norm_s1.astype(np.float32))
    # wavfile.write(save_dir+f"{ID}_1.wav", target_sr, norm_s2.astype(np.float32))
    

    # k += 1

    # print("score", scores)
    # plt.title(f"{best_time_corr}, {best_coch_corr}")
    # plt.scatter(comparison_corrs, scores, alpha=0.5, c='r', label=f"$\mu={np.mean(comparison_corrs)}$, $\sigma={np.std(comparison_corrs)}$")
    # plt.scatter(comparison_time_corrs, scores, alpha=0.5, c='b', label=f"$\mu={np.mean(comparison_time_corrs)}$, $\sigma={np.std(comparison_time_corrs)}$")
    # plt.legend()
    # plt.show()
    # input()
    # plt.clf()

    # clear_output(wait=False)


In [ ]:
print(len(clip_info))


all_coch_corrs = []
all_time_coch_corrs = []
all_scores = []

best_coch_corrs = []
best_time_coch_corrs = []
best_scores = []

for key in clip_info:
    curr_pair = clip_info[key]
    all_coch_corrs.extend(curr_pair["all_coch_corrs"])
    all_time_coch_corrs.extend(curr_pair["all_time_coch_corrs"])
    best_coch_corrs.append(curr_pair['best_coch_corr'])
    best_time_coch_corrs.append(curr_pair['best_time_corr'])

    all_scores.extend(curr_pair["scores"])
    best_scores.append(np.max(curr_pair["scores"]))

print(len(all_coch_corrs), len(all_time_coch_corrs))
print(len(best_coch_corrs), len(best_time_coch_corrs))
print(len(segment_starts))


x = plt.hist(all_coch_corrs, bins=100, color='r', alpha=0.5, density=True, label="All Coch Correlations")
y = plt.hist(best_coch_corrs, bins=100, color='m', alpha=0.5, density=True, label="Best Coch Correlations")


x = plt.hist(all_time_coch_corrs, bins=100, color='b', alpha=0.5, density=True, label="All Time-averaged Coch Correlations")
y = plt.hist(best_time_coch_corrs, bins=100, color='g', alpha=0.5, density=True, label="Best Time-averaged Coch Correlations")

plt.legend()
plt.title("Distribution of pairwise cochleagram correlations")
plt.ylabel("Density")
plt.xlabel("Pairwise Correlation")

In [ ]:
sound_idx = 1248
curr_pair = clip_info[list(clip_info.keys())[sound_idx]]
print(curr_pair.keys())

# graph is useful because i would EXCEPT to see 
## scores increase as pairwise time-average cochleagram correlations increase
## scores increase as pairwise cochleagram correlations decrease

plt.title(f"Scores")
plt.hist(curr_pair["scores"], label="All pairwise scores", bins=len(curr_pair["scores"]), alpha=0.5, color='r')
plt.axvline(np.max(curr_pair['scores']), color='b', alpha=0.5)

#plt.scatter(curr_pair["all_time_coch_corrs"], curr_pair["scores"], label="All pairwise time-averaged cochleagram correlations", alpha=0.5, color='b')
plt.ylabel("Density")
plt.xlabel("Score")


plt.legend()

In [ ]:
sound_idx = 1248
curr_pair = clip_info[list(clip_info.keys())[sound_idx]]
print(curr_pair.keys())

# graph is useful because i would EXCEPT to see 
## scores increase as pairwise time-average cochleagram correlations increase
## scores increase as pairwise cochleagram correlations decrease

plt.title(f"Best time-average coch corr {curr_pair['best_time_corr']:.2f} \n best coch corr {curr_pair['best_coch_corr']:.2f}")
plt.scatter(curr_pair["all_coch_corrs"], curr_pair["scores"], label="All pairwise cochleagram correlations", alpha=0.5, color='r')
plt.scatter(curr_pair["all_time_coch_corrs"], curr_pair["scores"], label="All pairwise time-averaged cochleagram correlations", alpha=0.5, color='b')
plt.ylabel("Score")
plt.xlabel("Pairwise-Correlation")
plt.xlim([-0.1, 1.1])

plt.axvline(curr_pair['best_time_corr'], color='b', alpha=0.5)
plt.axvline(curr_pair['best_coch_corr'], color='r', alpha=0.5)

plt.legend()

In [ ]:
best_coch_corr = []
best_time_corr = []

thres_coch = np.mean(all_coch_corrs) - np.std(all_coch_corrs)/2
thres_time = np.mean(all_time_coch_corrs) + np.std(all_time_coch_corrs)/2

print(thres_coch, thres_time)

filtered_clip_info = {}

for clip_id in clip_info.keys():
    clip_coch_corr = clip_info[clip_id]["best_coch_corr"]
    clip_time_corr = clip_info[clip_id]["best_time_corr"]

    if clip_coch_corr <= thres_coch and clip_time_corr >= thres_time:
    
        best_time_corr.append(clip_info[clip_id]["best_time_corr"])
        best_coch_corr.append(clip_info[clip_id]["best_coch_corr"])

        filtered_clip_info[clip_id] = copy.deepcopy(clip_info[clip_id])

print(len(filtered_clip_info))

In [ ]:
target_sr = 22050*2
segment_duration = 0.5
num_samples_per_segment = int(segment_duration * target_sr)
k = 0

for curr_sound in filtered_clip_info.keys():

    curr_pair = filtered_clip_info[curr_sound]


    
    # get audiopath
    audio_path = base_dir + curr_sound

    # grab the samples
    data, samplerate = librosa.load(audio_path)

    # 
    if samplerate != target_sr:
        data = librosa.resample(data, orig_sr=samplerate, target_sr=target_sr)

    # rms normalize
    data = rms_normalize(data)

    # first segment start time
    start_sample_1 = int(filtered_clip_info[curr_sound]['clip_1_onset']*target_sr)
    end_sample_1   = start_sample_1 + num_samples_per_segment
    segment_1 = rms_normalize(data[start_sample_1:end_sample_1])
    segment_1 = apply_linear_ramp(segment_1, target_sr)


    # first segment start time
    start_sample_2 = int(filtered_clip_info[curr_sound]['clip_2_onset']*target_sr)
    end_sample_2   = start_sample_2 + num_samples_per_segment 
    segment_2 = rms_normalize(data[start_sample_2:end_sample_2])
    segment_2 = apply_linear_ramp(segment_2, target_sr)

    display(ipd.Audio(segment_1, rate=target_sr))
    display(ipd.Audio(segment_2, rate=target_sr))

    ID = curr_sound.split("/")[-1].split(".")[0]

    # #print(save_dir+f"{ID}_0.wav")

    wavfile.write(save_dir+f"{ID}_0.wav", target_sr, segment_1.astype(np.float32))
    wavfile.write(save_dir+f"{ID}_1.wav", target_sr, segment_2.astype(np.float32))


    plt.title(f"Best time-average coch corr {curr_pair['best_time_corr']:.2f} \n best coch corr {curr_pair['best_coch_corr']:.2f}")
    plt.scatter(curr_pair["all_coch_corrs"], curr_pair["scores"], label="All pairwise cbochleagram correlations", alpha=0.5, color='r')
    plt.scatter(curr_pair["all_time_coch_corrs"], curr_pair["scores"], label="All pairwise time-averaged cochleagram correlations", alpha=0.5, color='b')
    plt.ylabel("Score")
    plt.xlabel("Pairwise-Correlation")
    plt.xlim([-0.1, 1.1])
    
    plt.axvline(curr_pair['best_time_corr'], color='b', alpha=0.5)
    plt.axvline(curr_pair['best_coch_corr'], color='r', alpha=0.5)
    
    plt.legend()
    plt.savefig(save_dir+f"{ID}.png")
    plt.clf()


    plt.title(f"Best time-average coch corr {curr_pair['best_time_corr']:.2f} \n best coch corr {curr_pair['best_coch_corr']:.2f}")
    plt.scatter(curr_pair["all_coch_corrs"], curr_pair["scores"], label="All pairwise cochleagram correlations", alpha=0.5, color='r')
    plt.scatter(curr_pair["all_time_coch_corrs"], curr_pair["scores"], label="All pairwise time-averaged cochleagram correlations", alpha=0.5, color='b')
    plt.ylabel("Score")
    plt.xlabel("Pairwise-Correlation")
    plt.xlim([-0.1, 1.1])
    
    plt.axvline(curr_pair['best_time_corr'], color='b', alpha=0.5)
    plt.axvline(curr_pair['best_coch_corr'], color='r', alpha=0.5)
    
    plt.legend()
    plt.savefig(save_dir+f"{ID}.png")
    plt.clf()

    plt.title(f"Scores")
    plt.hist(curr_pair["scores"], label="All pairwise scores", bins=len(curr_pair["scores"]), alpha=0.5, color='r')
    plt.axvline(np.max(curr_pair['scores']), color='b', alpha=0.5)
    
    #plt.scatter(curr_pair["all_time_coch_corrs"], curr_pair["scores"], label="All pairwise time-averaged cochleagram correlations", alpha=0.5, color='b')
    plt.ylabel("Density")
    plt.xlabel("Score")
    
    plt.savefig(save_dir+f"{ID}_scores.png")

    plt.legend()

    plt.clf()
    # k += 1
    
    #input()
    clear_output(wait=False)

In [ ]:
save_dir, curr_sound.split("/")[-1].split(".")[0]

In [ ]:
len(glob.glob(f"/om2/user/bjmedina/data/texture_pairs/cochleagram_corrs-full_clip-offset{offset}/*wav"))//2

In [ ]:
save_dir = "/om2/user/bjmedina/data/texture_pairs/spectral_contrast-stationarity-score{}/"

In [ ]:
# index of memory stimulus
k = 0
target_stationarity = -0.5


max_distances = []
# loop through all sounds in the eval set
for fp in fps:    
    stationarity, times = [], []

    #print(fp)
    curr_sound = fp
    audio_path = base_dir + curr_sound
    ipd.Audio(audio_path)
    
    stationarity, times = ss_scores_to_texture[curr_sound], times_to_texture[curr_sound]
    
    samplerate, data = wavfile.read(audio_path)
    try:
        data = data[:, 0].astype(np.float32)
    except:
        data = data.astype(np.float32)
    ipd.Audio(data, rate=samplerate)

    if len(stationarity) == 0:
        continue

    # Ideally, i think we should take all the sounds that have segments adjacent to each other that are stationary in time
    numbers = times
    #print(numbers)
    pairs = [(i, i+1) for i in range(len(numbers)-1) if abs(numbers[i] - numbers[i+1]) <= 1.0]
    
    #print(pairs)
    
    #TODO:
    # look for least stationary segment in clip, 
    # now that the whole clip is not stationary, find the two most different sections in that whole clip. 
    
    stationarity_scores = ss_scores_to_texture[curr_sound]
    
    min_mse = 1e9
    best_x = -1
    best_time = 0
    
    for x, time in enumerate(numbers): 
    
        mse = (target_stationarity - stationarity_scores[x]) ** 2
    
        if mse < min_mse:
            best_x = x
            best_time = time
            min_mse = mse
            
    #print(best_x, best_time)
    sound = data[int(best_time*samplerate):int((best_time+2)*samplerate)]
    
    wav_data = sound

    # grabbing all non-overlapping 0.5s segments from the 2s segment that has the stationarity closest to -0.6
    sr = samplerate  # Assume a common sample rate
    segment_duration = 0.5  # seconds
    
    # Assume `wav_data` contains the raw audio data
    # Assume `sr` is the sample rate of `wav_data`
    num_samples_per_segment = int(segment_duration * sr)
    num_segments = len(wav_data) // num_samples_per_segment
    
    # Split audio into non-overlapping 0.5s segments
    segments = [
        wav_data[i * num_samples_per_segment : (i + 1) * num_samples_per_segment]
        for i in range(num_segments)
    ]
    
    #print(num_segments)
    segment_onsets = [i*segment_duration for i in range(num_segments)]
    
    segment_onsets_relative_to_beginning = [(best_time+i*segment_duration) for i in range(num_segments)]
    #print(segment_onsets_relative_to_beginning)


    # Compute spectral contrast for each segment
    spectral_contrast_features = []
    for segment in segments: 
        S = np.abs(librosa.stft(segment))  # Compute magnitude spectrogram
        contrast = librosa.feature.spectral_contrast(S=S, sr=sr)  # Compute spectral contrast
        spectral_contrast_features.append(np.mean(contrast, axis=1))  # Take mean across time
    
    # Compute Euclidean distances between all pairs
    num_segments = len(spectral_contrast_features)
    max_distance = -1
    best_pair = (0, 1)
    
    for i in range(num_segments):
        for j in range(i + 1, num_segments):  # Ensure non-overlapping selection
            dist = euclidean(spectral_contrast_features[i], spectral_contrast_features[j])
            if dist > max_distance:
                max_distance = dist
                best_pair = (i, j)

    max_distances.append(max_distance)
    
    # # Extract the two most different segments
    # segment_1 = segments[best_pair[0]]
    # segment_2 = segments[best_pair[1]]
    
    # # Print results
    # #print(f"Most different segments: {segment_onsets_relative_to_beginning[best_pair[0]]} and {segment_onsets_relative_to_beginning[best_pair[1]]} with distance {max_distance}")
    
    # #print("best segments")

    # norm_s1 = rms_normalize(segment_1)
    # norm_s2 = rms_normalize(segment_2)

    # #display(ipd.Audio(norm_s1, rate=samplerate))
    # #display(ipd.Audio(norm_s2, rate=samplerate))

    # if not os.path.exists(save_dir.format(target_stationarity)):
    #     os.makedirs(save_dir.format(target_stationarity))

    # wavfile.write(save_dir.format(target_stationarity)+f"mem_stim_{k}_0.wav", samplerate, norm_s1.astype(np.float32))
    # wavfile.write(save_dir.format(target_stationarity)+f"mem_stim_{k}_1.wav", samplerate, norm_s2.astype(np.float32))

    k+=1

    #print(f"Sound {stationarity_scores[best_x]}:")
    #input()

In [ ]:
x = plt.hist(max_distances, bins=len(max_distances))
plt.axvline(x=np.mean(max_distances), c='r')

the_bar = np.mean(max_distances) + np.std(max_distances)

In [ ]:
# index of memory stimulus
k = 0

max_distances = []
# loop through all sounds in the eval set
for fp in fps:    
    stationarity, times = [], []

    #print(fp)
    curr_sound = fp
    audio_path = base_dir + curr_sound
    ipd.Audio(audio_path)
    
    stationarity, times = ss_scores_to_texture[curr_sound], times_to_texture[curr_sound]
    
    samplerate, data = wavfile.read(audio_path)
    try:
        data = data[:, 0].astype(np.float32)
    except:
        data = data.astype(np.float32)
    ipd.Audio(data, rate=samplerate)

    if len(stationarity) == 0:
        continue

    # Ideally, i think we should take all the sounds that have segments adjacent to each other that are stationary in time
    numbers = times
    #print(numbers)
    pairs = [(i, i+1) for i in range(len(numbers)-1) if abs(numbers[i] - numbers[i+1]) <= 1.0]
    
    #print(pairs)
    
    #TODO:
    # look for least stationary segment in clip, 
    # now that the whole clip is not stationary, find the two most different sections in that whole clip. 
    
    stationarity_scores = ss_scores_to_texture[curr_sound]
    
    min_mse = 1e9
    best_x = -1
    best_time = 0
    
    for x, time in enumerate(numbers): 
    
        mse = (target_stationarity - stationarity_scores[x]) ** 2
    
        if mse < min_mse:
            best_x = x
            best_time = time
            min_mse = mse
            
    #print(best_x, best_time)
    sound = data[int(best_time*samplerate):int((best_time+2)*samplerate)]
    
    wav_data = sound

    # grabbing all non-overlapping 0.5s segments from the 2s segment that has the stationarity closest to -0.6
    sr = samplerate  # Assume a common sample rate
    segment_duration = 0.5  # seconds
    
    # Assume `wav_data` contains the raw audio data
    # Assume `sr` is the sample rate of `wav_data`
    num_samples_per_segment = int(segment_duration * sr)
    num_segments = len(wav_data) // num_samples_per_segment
    
    # Split audio into non-overlapping 0.5s segments
    segments = [
        wav_data[i * num_samples_per_segment : (i + 1) * num_samples_per_segment]
        for i in range(num_segments)
    ]
    
    #print(num_segments)
    segment_onsets = [i*segment_duration for i in range(num_segments)]
    
    segment_onsets_relative_to_beginning = [(best_time+i*segment_duration) for i in range(num_segments)]
    #print(segment_onsets_relative_to_beginning)


    # Compute spectral contrast for each segment
    spectral_contrast_features = []
    for segment in segments: 
        S = np.abs(librosa.stft(segment))  # Compute magnitude spectrogram
        contrast = librosa.feature.spectral_contrast(S=S, sr=sr)  # Compute spectral contrast
        spectral_contrast_features.append(np.mean(contrast, axis=1))  # Take mean across time
    
    # Compute Euclidean distances between all pairs
    num_segments = len(spectral_contrast_features)
    max_distance = -1
    best_pair = (0, 1)
    
    for i in range(num_segments):
        for j in range(i + 1, num_segments):  # Ensure non-overlapping selection
            dist = euclidean(spectral_contrast_features[i], spectral_contrast_features[j])
            if dist > max_distance:
                max_distance = dist
                best_pair = (i, j)

    #max_distances.append(max_distance)

    if max_distance <= the_bar:
        continue
    
    # Extract the two most different segments
    segment_1 = segments[best_pair[0]]
    segment_2 = segments[best_pair[1]]
    
    # Print results
    #print(f"Most different segments: {segment_onsets_relative_to_beginning[best_pair[0]]} and {segment_onsets_relative_to_beginning[best_pair[1]]} with distance {max_distance}")
    
    #print("best segments")

    norm_s1 = rms_normalize(segment_1)
    norm_s2 = rms_normalize(segment_2)

    #display(ipd.Audio(norm_s1, rate=samplerate))
    #display(ipd.Audio(norm_s2, rate=samplerate))

    if not os.path.exists(save_dir.format(target_stationarity)):
        os.makedirs(save_dir.format(target_stationarity))

    wavfile.write(save_dir.format(target_stationarity)+f"{ID}_0.wav", samplerate, norm_s1.astype(np.float32))
    wavfile.write(save_dir.format(target_stationarity)+f"{ID}_1.wav", samplerate, norm_s2.astype(np.float32))

    k+=1

    #print(f"Sound {stationarity_scores[best_x]}:")
    #input()

In [ ]:
# Compute spectral contrast for each segment
spectral_contrast_features = []
for segment in segments:
    S = np.abs(librosa.stft(segment))  # Compute magnitude spectrogram
    contrast = librosa.feature.spectral_contrast(S=S, sr=sr)  # Compute spectral contrast
    spectral_contrast_features.append(np.mean(contrast, axis=1))  # Take mean across time

# Compute Euclidean distances between all pairs
num_segments = len(spectral_contrast_features)
max_distance = -1
best_pair = (0, 1)

for i in range(num_segments):
    for j in range(i + 1, num_segments):  # Ensure non-overlapping selection
        dist = euclidean(spectral_contrast_features[i], spectral_contrast_features[j])
        if dist > max_distance:
            max_distance = dist
            best_pair = (i, j)

# Extract the two most different segments
segment_1 = segments[best_pair[0]]
segment_2 = segments[best_pair[1]]

# Print results
print(f"Most different segments: {segment_onsets_relative_to_beginning[best_pair[0]]} and {segment_onsets_relative_to_beginning[best_pair[1]]} with distance {max_distance}")

print("best segments")
display(ipd.Audio(segment_1, rate=samplerate))
display(ipd.Audio(segment_2, rate=samplerate))


In [ ]:
# is the code doing what it is supposed to be doing?
# check other pairs
print(best_pair)
for i in range(num_segments):
    for j in range(i, num_segments):
        if i == j:
            continue
        if i == best_pair[0] and j == best_pair[1]:
            continue
        other_segment_1 = segments[i]
        other_segment_2 = segments[j]
        print(f"segment {i} and {j}")
        display(ipd.Audio(other_segment_1, rate=samplerate))
        display(ipd.Audio(other_segment_2, rate=samplerate))

# what are some selection criteria for screening for discriminable stimuli (so that I don't have to listen to all of them)?

# 1)
# - screen each 10 sec clip for the 2 sec seg that has the stationarity score most similar to -0.6. 
# - get that clip. take the first 0.5s, then take the adjacent 0.5s. (or first 0.5 or some other 0.5)
# - maybe look for the most spectrally different section

# 2) 
# - do same as 1 but chose sound with the lowest stationarity score (or try different scores like -0.5)